# ACCESS-A-WEB

Aquest notebook executa dos scripts:
1. `accdb_a_sqlite.py` – converteix una base de dades Microsoft Access (.accdb/.mdb) a SQLite.
2. `genera_app_flask.py` – crea una aplicació web Flask personalitzada per gestionar la base de dades SQLite resultant.

Arxius mínims que has de tenir al directori del projecte:
- `accdb_a_sqlite.py`
- `genera_app_flask.py`
- `LLEGEIX-ME.md`
- `ACCESS-A-WEB.ipynb`
- `requirements.txt`
- Arxiu `*.accdb` o `*.mdb` amb la base de dades per transformar
- (OPCIONAL) `logo.png` amb el logo que vols mostrar a l'app web

#### **Important:** Abans d'executar aquest notebook, has de seguir les instruccions de l'arxiu `LLEGEIX-ME.md`.

## 1. Configuració inicial

Modifica els valors següents en funció de com s'anomeni la teva base de dades i, si s'escau, el teu logo. El logo només pot ser una imatge PNG o JPG. Aquest notebook està pensat perquè només hagis de modificar el codi de la primera cel·la. 

Només s'ha comentat el codi de la primera cel·la perquè és la que està pensada per persones no avesades a la programació. Per la resta de cel·les, s'ha considerat que el codi és suficientment autoexplicatiu per a persones habituades a la programació i només s'ha fet la separació per blocs d'accions.

In [ ]:
# Ruta a l'arxiu Access d'entrada (ex: "C:/dades/la_meva_base.accdb" o "./dades/base_dades.mdb")
INPUT_ACCDB = "Northwind.accdb" # Obligatori

# Ruta on es crearà la base de dades SQLite (ex: "./base_convertida.sqlite")
OUTPUT_SQLITE = "Northwind.sqlite" # Opcional

# Carpeta on es generarà l'aplicació Flask
FLASK_OUTPUT_DIR = "app_web" # Opcional

# Títol que es mostrarà en l'app web que es crearà
APP_TITLE = "Gestor de base de dades" # Opcional

# Imatge que es mostrarà en l'app web que es crearà
LOGO_PATH = "logo.png" # Opcional

## 2. Configuració per defecte i resum

Si no s'ha definit una configuració explícita dels paràmetres opcionals, s'assigna una configuració per defecte. 

In [ ]:
from pathlib import Path

if not OUTPUT_SQLITE:
    OUTPUT_SQLITE = f"{Path(INPUT_ACCDB).stem}.sqlite"  
if not FLASK_OUTPUT_DIR:
    FLASK_OUTPUT_DIR = "app_flask_sqlite"
if not APP_TITLE:
    APP_TITLE = "Gestor de Base de Dades"
if not LOGO_PATH:
    LOGO_PATH = None

print("\n--- Configuració inicial ---")
print(f"Access: {INPUT_ACCDB}")
print(f"SQLite: {OUTPUT_SQLITE}")
print(f"App Flask: {FLASK_OUTPUT_DIR}")
print(f"Títol: {APP_TITLE}")
print(f"Logo: {LOGO_PATH if LOGO_PATH else '(cap logo)'}")

## 3. Instal·lació de dependències per Linux/macOS

S'instal·la el paquet `mdbtools` perquè les dependències de requirements.txt no l'inclouen.

In [ ]:
import sys, subprocess, platform

if platform.system() != "Windows":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mdbtools-python"])

if platform.system() != "Windows":
    try:
        subprocess.run(["mdb-tables", "--version"], capture_output=True, check=True)
        print("✓ mdbtools detectat correctament.")
    except (subprocess.SubprocessError, FileNotFoundError):
        print("⚠️ ATENCIÓ: No s'ha trobat 'mdb-tools'. Instal·la mdbtools:")
        if platform.system() == "Linux":
            print("   sudo apt update && sudo apt install mdbtools")
        elif platform.system() == "Darwin":
            print("   brew install mdbtools")

## 4. Execució del primer script: Access → SQLite

S'executa `accdb_a_sqlite.py` amb els paràmetres definits.

In [ ]:
import os, sys

script1 = "accdb_a_sqlite.py"
if not os.path.isfile(script1):
    raise FileNotFoundError(f"No es troba el fitxer '{script1}' al directori actual.")

if not os.path.isfile(INPUT_ACCDB):
    raise FileNotFoundError(f"No es troba el fitxer Access: {INPUT_ACCDB}")

cmd1 = [sys.executable, script1, "-i", INPUT_ACCDB, "-o", OUTPUT_SQLITE, "-v"]
print("Executant:", " ".join(cmd1))
result1 = subprocess.run(cmd1)
if result1.returncode != 0:
    raise RuntimeError("Error durant la conversió Access → SQLite")
print("\n✅ Conversió completada. SQLite creat a:", OUTPUT_SQLITE)

## 5. Execució del segon script: Generació de l'app web

Es genera una aplicació web completa per gestionar la base de dades SQLite que s'ha transformat.

**Usuaris predefinits:** (USUARI / PASSWORD)  
- admin-01 / admin123   -->   (rol d'administrador)
- editor-01 / editor123   -->   (rol d'editor)
- lector-01 / lector123   -->   (rol de lector)

Podràs modificar-los després des de la pròpia interfície web entrant amb l'usuari administrador.

In [ ]:
import shutil
import secrets

script2 = "genera_app_flask.py"
if not os.path.isfile(script2):
    raise FileNotFoundError(f"No es troba el fitxer '{script2}' al directori actual.")

cmd2 = [sys.executable, script2, "--db-path", OUTPUT_SQLITE, "--output-dir", FLASK_OUTPUT_DIR, "--title", APP_TITLE]
if LOGO_PATH:
    if not os.path.isfile(LOGO_PATH):
        print(f"⚠️ Avís: el logo '{LOGO_PATH}' no existeix. S'ignorarà.")
    else:
        cmd2.extend(["--logo", LOGO_PATH])
print("Executant:", " ".join(cmd2))
result2 = subprocess.run(cmd2)
if result2.returncode != 0:
    raise RuntimeError("Error durant la generació de l'aplicació Flask")
print("\n✅ Aplicació Flask generada correctament a la carpeta:", FLASK_OUTPUT_DIR)

origen = OUTPUT_SQLITE
desti = FLASK_OUTPUT_DIR
try:
    shutil.copy2(origen, desti)    
except Exception as e:
    print(f"❌ Error inesperat: {e}") 
else:
    print("✅ Base de dades copiada i sobreescrita correctament.")

SECRET_KEY = secrets.token_hex(32)
fitxer = FLASK_OUTPUT_DIR +"/app.py"
text_antic = "CLAU-SECRETA"
text_nou = SECRET_KEY
try:
    with open(fitxer, "r", encoding="utf-8") as f:
        contingut = f.read()
    if text_antic in contingut:
        nou_contingut = contingut.replace(text_antic, text_nou)
        with open(fitxer, "w", encoding="utf-8") as f:
            f.write(nou_contingut)        
        print("✅ Clau secreta afegida correctament.")
    else:
        print(f"❌ No s'ha trobat el text '{text_antic}' al fitxer {fitxer}. No s'ha realitzat cap canvi.")
except FileNotFoundError:
    print(f"❌ Error: No es troba el fitxer '{fitxer}'.")
except Exception as e:
    print(f"❌ Error inesperat: {e}")


## 6. Instruccions per arrencar l'aplicació web

L'aplicació generada és completament autònoma. Segueix aquests passos per posar-la en marxa:

In [ ]:
print("="*60)
print("INSTRUCCIONS D'EXECUCIÓ")
print("="*60)
print(f"\n1. En una terminal de VSC, accedeix a la carpeta de l'aplicació:")
print(f"   cd {FLASK_OUTPUT_DIR}")
print("\n2. Des de la terminal de VSC, arrenca el servidor Flask:")
print("   python app.py")
print("\n3. Obre el teu navegador a: http://localhost:5000")
print("\n   Credencials per defecte:")
print("   - administrador: admin01 / admin123")
print("   - editor:        editor01 / editor123")
print("   - lector:        lector01 / lector123")
print("\n" + "="*60)